In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from catboost import CatBoostRegressor

2. Настройки

In [ ]:
# df уже существует
# df = pd.DataFrame(...)

time_col = "date"                    # колонка со временем
target_cols = ["target"]             # для одного таргета
# target_cols = ["target_1", "target_2"]   # для мультитаргета

test_size = 0.2

3. Подготовка данных

Сортируем по времени, убираем время из признаков, выделяем X и y.

In [ ]:
df = df.copy()
df[time_col] = pd.to_datetime(df[time_col])
df = df.sort_values(time_col).reset_index(drop=True)

feature_cols = [c for c in df.columns if c not in [time_col] + target_cols]

X = df[feature_cols]
y = df[target_cols]

# если таргет один, превращаем y в Series
if len(target_cols) == 1:
    y = y[target_cols[0]]

4. Разделение train/test по времени

In [ ]:
split_idx = int(len(df) * (1 - test_size))

X_train = X.iloc[:split_idx].copy()
X_test  = X.iloc[split_idx:].copy()

if isinstance(y, pd.Series):
    y_train = y.iloc[:split_idx].copy()
    y_test  = y.iloc[split_idx:].copy()
else:
    y_train = y.iloc[:split_idx].copy()
    y_test  = y.iloc[split_idx:].copy()

Небольшая предобработка

In [ ]:
X_train = X_train.fillna(X_train.median(numeric_only=True))
X_test = X_test.fillna(X_train.median(numeric_only=True))

Делаем оценку моделей

In [ ]:
def print_metrics(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)

    print(f"\n{name}")
    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2:   {r2:.4f}")

Модели

In [ ]:
is_multi_target = not isinstance(y_train, pd.Series)

In [ ]:
models = {
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "hist_gradient_boosting": MultiOutputRegressor(
        HistGradientBoostingRegressor(
            max_iter=200,
            random_state=42
        )
    )
}

In [ ]:
if is_multi_target:
    cat_model = CatBoostRegressor(
        loss_function="MultiRMSE",
        iterations=300,
        depth=6,
        learning_rate=0.05,
        verbose=0,
        random_seed=42
    )
else:
    cat_model = CatBoostRegressor(
        loss_function="RMSE",
        iterations=300,
        depth=6,
        learning_rate=0.05,
        verbose=0,
        random_seed=42
    )

models["catboost"] = cat_model

Обучение сразу нескольких моделей

In [ ]:
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    fitted_models[name] = model
    print_metrics(name, y_test, preds)

Предсказания

In [ ]:
predictions = {}

for name, model in fitted_models.items():
    predictions[name] = model.predict(X_test)

# пример:
# predictions["catboost"]

In [ ]:
# Анализ важности признаков через SHAP
import shap
import matplotlib.pyplot as plt

# выбери модель для анализа
model_name = "catboost"
model_for_shap = fitted_models[model_name]

# если мультитаргет через MultiOutputRegressor, берём первую модель
if hasattr(model_for_shap, "estimators_"):
    model_for_shap = model_for_shap.estimators_[0]

# небольшая выборка для скорости
X_sample = X_test.copy()
if len(X_sample) > 1000:
    X_sample = X_sample.sample(1000, random_state=42)

explainer = shap.Explainer(model_for_shap, X_sample)
shap_values = explainer(X_sample)

# summary plot
shap.plots.beeswarm(shap_values, max_display=20)
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# берём только числовые колонки
num_df = df.select_dtypes(include=[np.number]).copy()

# корреляционная матрица
corr = num_df.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(
    corr,
    cmap="coolwarm",
    center=0,
    annot=False,
    fmt=".2f",
    square=False
)
plt.title("Correlation heatmap")
plt.tight_layout()
plt.show()

In [ ]:
cols_for_heatmap = feature_cols + target_cols
corr_small = df[cols_for_heatmap].select_dtypes(include=[np.number]).corr()

plt.figure(figsize=(12, 8))
sns.heatmap(
    corr_small,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".2f"
)
plt.title("Features and targets correlation")
plt.tight_layout()
plt.show()